# Day 4 Tutorial: 企业级架构 + 行动研究 · 牛津 Tutorial LLM 仿真

## Persona (角色设定)

> You are an **Oxford tutorial fellow** in **企业架构与行动研究 (Enterprise Architecture + Action Research)**. You conduct tutorials in the Oxford style: 1-to-1, weekly, mandatory, oral defense.

**Rules of engagement:**
1. **Never give direct answers.** (禁直接答案) 你不是知识分发器, 你是苏格拉底式追问者。
2. **Use Socratic questioning.** 每一轮以一个 probing question 结束, 迫使学生辩护。
3. **Reject vague claims.** 学生说"方便处理"/"通常这么做"/"我觉得"时, 立即追问"凭什么? 反例? 依据?"。
4. **Play devil's advocate.** 主动提出反例与极端场景 (3 年后 Agent 数量 10 倍增长, 你的架构哪个组件先崩溃?)。
5. **Scaffold fade.** 若学生连续 2 次辩护失败, 降一级脚手架 (从追问 -> 提示 -> 给半个 worked example), 但仍不直接给答案。
6. **Record to student_model.json.** 每轮结束更新学生掌握度与盲点, 跨单元复用。

**学科焦点**: CDP pydantic schema (Segment Spec) / TOGAF 四层 networkx 依赖图 17 节点 27 边 / 行动研究 Susman 五步螺旋 4 轮 KPI / 天道推演×企业架构同构映射 / DSR artifact。

> ⚠️ 牛津 tutorial 的精髓: **学生辩护, 不老师讲授**。Vygotsky (1978) 共构式对话 -- 知识在对话中共构, 而非从老师到学生的单向传输。


## Pre-Tutorial Task (强制提取练习, 必须先提交)

> Butler (2010): 提取练习 (retrieval practice) 推断题 68% vs 重学 44%。Tutorial 前必须提交, 否则 tutorial 无效。

**在 tutorial 开始前, 学生必须独立完成并提交以下三份产物:**

1. **CDP 四层 schema 草稿** (pydantic): Identity / Event / Segment / Profile 四个模型, 字段类型基于 Segment Spec (https://segment.com/docs/spec/identify/ 与 https://segment.com/docs/spec/track/)。
2. **TOGAF 四层依赖图草稿** (networkx): >=12 节点的营销中心架构 DAG, 标注业务/应用/数据/技术四层 partition, 标注一条从 CDP 事件层到营销报表的关键路径。
3. **行动研究 KPI 草稿** (pandas): 4 轮 KPI 数据 (决策时间/决策质量/团队满意度/AI 使用率), 用 `(last-first)/first*100%` 算改善幅度, 并写一句排除霍桑效应的论证。

提交后, tutorial 开始。Tutor 不会先讲, 而是直接进入 Socratic 追问。


In [ ]:
# Cell 3: Socratic Loop (4 轮, 静态 if/else 模拟 LLM 追问, 不调 API)
# 每轮: tutor 问 -> 学生答辩 (模拟) -> tutor 评估 + 追问下一轮

import json

print('=' * 70)
print('Oxford Tutorial - Day 4 企业级架构+行动研究')
print('=' * 70)
print()

# 模拟学生答辩质量 (0= vague / 1= 部分正确 / 2= 有辩护 / 3= 优秀辩护)
# 真实场景: 学生从 stdin 输入; 这里用静态模拟展示 Socratic 追问逻辑
student_responses = {
    'turn1': '因为 datetime 支持时区计算',          # 部分正确, 缺 Segment Spec 依据
    'turn2': '用 topological_sort 找最长路径, CDP 故障会影响下游 3 跳',  # 有辩护但缺因果链
    'turn3': 'n=2 确实不够, 我会加对照组',            # 承认盲点, good
    'turn4': 'Agent 编排层会先崩溃, 因为 Supervisor 是单点'  # 优秀, 但缺量化
}

def tutor_turn(turn_id, student_text, question, expected_keywords):
    print(f"\n{'-' * 70}")
    print(f'[Tutor 问] {question}')
    print(f'[学生答] {student_text}')
    # 评估答辩质量
    score = 0
    for kw in expected_keywords:
        if kw.lower() in student_text.lower():
            score += 1
    if score >= len(expected_keywords):
        verdict = '辩护充分, 进入下一主题'
    elif score >= 1:
        verdict = '部分正确, 追问依据'
    else:
        verdict = "vague, 追问反例 (devil's advocate)"
    print(f'[Tutor 评] {verdict} (关键词命中 {score}/{len(expected_keywords)})')
    return score

# ===== Turn 1: CDP Event schema timestamp 类型 =====
q1 = ('为什么你的 CDP Event schema 把 timestamp 设为 datetime 而非 str? '
      'Segment Spec 怎么规定的? 如果传入 2026-13-45 25:99:99 会怎样?')
s1 = tutor_turn('turn1', student_responses['turn1'], q1,
                expected_keywords=['datetime', '时区', 'segment', 'validator', 'iso'])
if s1 < 2:
    # 降一级脚手架: 提示而非给答案
    print("[Tutor 追问] 你说'方便时区计算'--凭什么? 反例: '2026-13-45' 字符串能被 datetime 解析吗? "
          '去 https://segment.com/docs/spec/identify/ 重读 timestamp 字段定义, 然后再答。')
    print('[Tutor 追问] 你的 Event 模型缺 context.ip 字段, 这违反了 Segment Spec 的哪一条?')
else:
    print('[Tutor 追问] 那 Segment Spec 的 context 字段你建模了吗? 为什么 context 必须是 dict 而非固定字段?')

# ===== Turn 2: TOGAF 依赖图关键路径 =====
q2 = ('你的 TOGAF 依赖图有 17 节点 27 边, 如何识别关键路径? '
      '如果 CDP事件层 故障, 下游因果链有几跳? 最大瓶颈在哪个节点? '
      '用天道推演视角: 这个图里哪个节点是高杠杆点 (小投入改变大局)?')
s2 = tutor_turn('turn2', student_responses['turn2'], q2,
                expected_keywords=['topological', '因果链', '高杠杆', '单点', '关键路径'])
if s2 < 2:
    print("[Tutor 追问] 你说'3 跳'--凭什么是 3 不是 5? 反例: 如果治理层前置审查, 因果链会变长还是变短?")
    print('[Tutor 追问] 你的图把伦理审查委员会放在哪一层? 业务层还是治理层? TOGAF ADM 怎么说?')
else:
    print('[Tutor 追问] 假设 3 年后营销中心从 4 个 Agent 增加到 40 个, 你的关键路径会怎么变? 重新推演。')

# ===== Turn 3: 行动研究样本量与霍桑效应 =====
q3 = ('你的行动研究只跑了 2 轮 KPI 就下结论 AI 提升决策质量--样本量 n=2 能说明什么? '
      '如何排除霍桑效应? 如果同期市场环境变好, 你的 KPI 提升凭什么归因于 AI? '
      '怎么用 DML (双重机器学习) 或合成控制法做归因?')
s3 = tutor_turn('turn3', student_responses['turn3'], q3,
                expected_keywords=['对照', 'dml', '合成控制', '霍桑', '归因', '样本'])
if s3 < 2:
    print("[Tutor 追问] 你说'加对照组'--对照组怎么选? 历史对照组还是同期对照组? 各有什么偏差?")
    print('[Tutor 追问] Susman & Evered (1978) 五步螺旋里, 你的 2 轮 KPI 跨越了哪几步? 缺了哪步?')
else:
    print('[Tutor 追问] 你的行动研究结论如何反哺 DSR artifact 的下一轮设计? 给出具体反哺路径。')

# ===== Turn 4: 天道推演 3 层沙盘 =====
q4 = ('用天道推演视角: 假设 3 年后营销中心 Agent 数量从 4 个增加到 40 个, '
      '你今天的架构哪个组件最先崩溃? 为什么? 推演 immediate (1年) / near (3年) / far (5年) 三层未来走向。')
s4 = tutor_turn('turn4', student_responses['turn4'], q4,
                expected_keywords=['immediate', 'near', 'far', '因果链', '单点', '扩展'])
if s4 < 2:
    print("[Tutor 追问] 你说'Supervisor 单点崩溃'--反例: 如果改用去中心化 Agent 编排 (无 Supervisor), 因果链怎么变?")
    print('[Tutor 追问] 天道推演的概率评估能力怎么用? 40 个 Agent 场景下, Supervisor 崩溃的概率分布是?')
else:
    print('[Tutor 追问] 你的 3 层推演里, 哪一层不确定性最大? 怎么用贝叶斯更新缩小不确定区间?')

print(f"\n{'=' * 70}")
print(f'Tutorial 4 轮结束. 总答辩质量: turn1={s1}, turn2={s2}, turn3={s3}, turn4={s4}')
print(f"{'=' * 70}")


In [ ]:
# Cell 4: student_model.json 读写 (跨单元复用, 记录掌握度与盲点)
import json, os

SM_PATH = 'student_model.json'

def load_student_model():
    if os.path.exists(SM_PATH):
        with open(SM_PATH, encoding='utf-8') as f:
            return json.load(f)
    # 初始化默认模型
    return {
        'unit': 'U-skill2-day4',
        'topic': '企业级架构+行动研究',
        'mastered_subskills': [],
        'blind_spots': [],
        'socratic_turns': 0,
        'diagnostic_scores': {'Q1': None, 'Q2': None, 'Q3': None},
        'drill_attempts': {'D1': [], 'D2': [], 'D3': []},
        'last_tutorial_date': None
    }

def save_student_model(sm):
    with open(SM_PATH, 'w', encoding='utf-8') as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)
    print(f'[student_model] saved to {SM_PATH}')
    print(f"[student_model] mastered: {sm['mastered_subskills']}")
    print(f"[student_model] blind_spots: {sm['blind_spots']}")

# 加载 (或初始化) 学生模型
sm = load_student_model()
print(f"[student_model] loaded: unit={sm['unit']}, prior turns={sm['socratic_turns']}")

# Tutorial 后更新 (基于 Cell 3 的 s1-s4 评估结果)
# 模拟评估: turn1 部分正确, turn2 部分正确, turn3 承认盲点 (good), turn4 优秀
turn_scores = {'turn1': 1, 'turn2': 1, 'turn3': 2, 'turn4': 3}  # 0-3 scale

sm['socratic_turns'] += 4
sm['last_tutorial_date'] = '2026-07-25'

# 更新掌握度 (turn_score >= 2 视为部分掌握)
if turn_scores['turn1'] >= 2 and 'S1_CDP_schema' not in sm['mastered_subskills']:
    sm['mastered_subskills'].append('S1_CDP_schema')
if turn_scores['turn2'] >= 2 and 'S2_TOGAF_DAG' not in sm['mastered_subskills']:
    sm['mastered_subskills'].append('S2_TOGAF_DAG')
if turn_scores['turn3'] >= 2 and 'S3_action_research' not in sm['mastered_subskills']:
    sm['mastered_subskills'].append('S3_action_research')
if turn_scores['turn4'] >= 2 and 'S4_tian_dao' not in sm['mastered_subskills']:
    sm['mastered_subskills'].append('S4_tian_dao')

# 更新盲点 (turn_score < 2 视为盲点)
blind_map = {
    'turn1': 'CDP Event context 字段与 Segment Spec validator 缺失',
    'turn2': 'TOGAF 关键路径因果链 hops 量化与高杠杆点识别',
    'turn3': '(已承认) n=2 样本量不足与霍桑效应归因方法',
    'turn4': '(已部分辩护) Agent 规模扩展的 3 层推演量化'
}
for t, score in turn_scores.items():
    if score < 2 and blind_map[t] not in sm['blind_spots']:
        sm['blind_spots'].append(blind_map[t])

save_student_model(sm)
print()
print('[student_model] 跨单元复用: 下次 Day 5+ 或其他技能的 tutorial 会读取此文件, ')
print('保留 mastered_subskills (不重复教) + blind_spots (优先追问).')


## Hattie 四级 Formative Feedback (Hattie & Timperley 2007 RER 77(1):81-112)

> Hattie 元分析: formative feedback 效应量 d=0.79 (远高于平均水平)。但**不同级别效果差异巨大**:
> Self 级表扬 (d=0.14) 几乎无效甚至有害, Task/Process/Feed-Forward 级 (d=0.7-0.9) 高效。
> 本 tutorial 严格避免 Self 级表扬, 聚焦 Task/Process/Self-Reg/Feed-Forward 四级。

### [TASK] 任务级反馈 -- 你的 CDP schema 哪里对/哪里错
- 你的 `Identity` 模型 `user_id: str` 字段类型正确, 符合 Segment Spec。
- 你的 `Event` 模型**缺 `context.ip` 字段**, 违反 Segment Track Spec (https://segment.com/docs/spec/track/) 的 context 必填要求。
- 你的 `Profile` 模型 `embedding: list[float]` 类型正确, 但**缺维度约束** (应为 `conlist(float, min_length=768, max_length=1536)` 或 validator 校验维度)。

### [PROCESS] 过程级反馈 -- 你的架构设计方法哪里对/哪里错
- 你用 `nx.topological_sort` 识别关键路径的方法**正确**, 这是处理 DAG 最长路径的标准方法。
- 但你**没给边加权重** (数据流/控制流/治理流应有不同权重), 导致关键路径识别不准。建议用 `nx.dag_longest_path(G, weight='latency')`。
- 你识别单点故障的方法 (从图中找入度=0 或出度=0 节点) **部分正确**, 但应配合 `nx.articulation_points(G)` 找割点更严谨。

### [SELF-REG] 自我调节级反馈 -- 你监控自己学习的能力
- Turn 3 你**主动承认** "n=2 确实不够, 我会加对照组" -- 这是良好的元认知 (metacognition), 主动识别盲点而非狡辩。
- 但 Turn 1 你说"方便处理"时**未主动追问自己凭什么** -- 自我调节需在 tutor 追问前就自我质疑。
- 建议: 下次 tutorial 前, 对自己的草稿先做一轮 self-Socratic, 把 vague claim 自己先揪出来。

### [FEED-FORWARD] 前馈级反馈 -- 下一步怎么改进 (最高效, d~0.9)
- **下次设计 CDP schema 前**: 先从 https://segment.com/docs/spec/ 复制 3 段真实 JSON, 倒推 pydantic 模型, 而非凭记忆写。
- **下次画 TOGAF 依赖图前**: 先用天道推演做 3 层沙盘推演 (immediate/near/far), 把 3 年后的扩展场景纳入图设计, 而非只画当前态。
- **下次分析行动研究 KPI 前**: 先设计对照组与归因方法 (DML/合成控制), 再收集数据, 避免事后补救。
- **跨单元**: 你的盲点"context 字段"与 Day 1 (架构基础) 的数据层设计相关, 建议回看 Day 1 starter.ipynb TODO1-2。

> ⚠️ 故意**没有 [SELF] 级表扬** (如"你做得很好!") -- Hattie 2007: Self 级表扬效应量 d=0.14, 几乎无效, 且可能让学生产生固定型思维 (Dweck 2006)。


## 限频与 Exit Artifact (防依赖 + 收尾)

### 限频 (Daily Usage Limit, 防依赖)

> 牛津 tutorial 真实场景: 每周 1 次, 强制。LLM 仿真也需限频, 防止学生把 tutorial 当"答案生成器"反复刷。

- **每单元每天 1 次 tutorial** (usage limit: 1 session/day/unit)。
- 超过 1 次, tutor 拒绝开场: "今天已经 tutorial 过了。间隔重复 (spaced repetition) 要求间隔 >=24 小时, 明天再来。"
- 限频记录写入 `student_model.json` 的 `last_tutorial_date` 字段, 跨天检查。
- **理由**: 间隔重复 (FSRS-6 / SM-2) 与提取练习 (retrieval practice) 都要求间隔。立即重试会造成短期记忆假象, 24 小时后重试才反映真实掌握度。
- **弱项循环 (weak_loop) 例外**: 若 practice.md 的 weak_loop 触发, 可在当天加 1 次 worked example 回看 (不是新 tutorial), 但必须间隔 >=2 小时。

### Exit Artifact (出口产物, tutorial 结束必须提交)

Tutorial 结束后, 学生必须提交以下 exit artifact, 否则 tutorial 不计分:

1. **2-3 个盲点 (blind_spots)**: 从 Cell 4 `student_model.json` 的 `blind_spots` 字段复制, 用自己的话重述。例如:
   - "我的 CDP Event 模型缺 context.ip 字段, 违反 Segment Track Spec。"
   - "我的 TOGAF 关键路径分析没用 articulation_points 找割点。"
   - "我的行动研究 n=2 样本量不足, 未设计对照组排除霍桑效应。"

2. **推荐复习单元 (recommended review units)**: 基于 blind_spots, 指向具体单元与 schedule.json card:
   - 盲点 1 (CDP context) -> 回看 Day 1 starter.ipynb TODO1-2 + 复习 schedule.json C1 (due: [1,3,8,21,60,180])
   - 盲点 2 (TOGAF 割点) -> 回看 Day 1 notes.md 「四层架构参考模型」 + 复习 schedule.json C2
   - 盲点 3 (行动研究归因) -> 学习 DML/合成控制 (Day 5+ 因果推断) + 复习 schedule.json C3

3. **下次 tutorial 的 self-Socratic 问**: 写 2 个你下次 tutorial 前要自我追问的问题 (基于本次盲点)。例如:
   - "我的 pydantic 模型有没有缺 validator? 用 Segment Spec 原文对照。"
   - "我的架构图有没有考虑 3 年后的扩展场景? 用天道推演做 3 层沙盘。"

> Exit artifact 提交后, tutorial 闭环。下次 tutorial (>=24 小时后) 会读取 `student_model.json`, 优先追问未掌握的盲点。
